# Gemini API を呼び出す (無料枠)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/gemini_basic.ipynb)

NLP クックブック 13-13 ／ 大規模言語モデル (LLM)

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

Gemini を API で使いたい。システムプロンプト、温度、対話の履歴などを細かく指定したい。Colab のシークレット機能で API キーを安全に扱いたい。

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

# Google AI Studio (aistudio.google.com) で API キーを作り、左の 🔑 (シークレット) に GOOGLE_API_KEY として登録し、
# 「ノートブックからのアクセス」を有効にしておく (シークレットの画面の「Gemini API キーをインポート」からも登録できる)
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# 使えるモデルを確認する (モデル名の入れ替わりが早いので、決め打ちせずに一覧から選ぶ)
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []) and "flash" in m.name:
        print(m.name, m.input_token_limit, m.output_token_limit)

In [ ]:
from google.genai import types

MODEL = "gemini-flash-latest"      # 最新の Flash モデルを指す別名。上の一覧の名前 (models/ を除いた部分) を直接指定してもよい

response = client.models.generate_content(
    model=MODEL,
    contents="BPE と WordPiece の違いを 3 行で説明してください。",
    config=types.GenerateContentConfig(
        system_instruction="あなたは自然言語処理の専門家です。簡潔に答えてください。",
        temperature=0.2,
    ),
)
print(response.text)
u = response.usage_metadata
print(u.prompt_token_count, "入力トークン /", u.candidates_token_count, "出力トークン / 思考", u.thoughts_token_count)

In [ ]:
# 対話を続ける: chats を使うと、履歴を自動で保持して毎回送ってくれる
chat = client.chats.create(model=MODEL, config=types.GenerateContentConfig(system_instruction="日本語教師として、やさしく答えてください。"))
print(chat.send_message("「は」と「が」の違いを一言で。").text)
print(chat.send_message("例文を 1 つずつ挙げてください。").text)
print(len(chat.get_history()), "件の発話が履歴にある")

- API キーをノートブックに直接書かない。共有や GitHub への push で漏れる。
- Gemini API には無料枠があり、Flash 系のモデルは料金なしで使える (2026 年時点)。ただし、1 分あたり・1 日あたりのリクエスト数に上限がある。上限は AI Studio の画面でプロジェクトごとに確認できる。
- 無料枠で送った入力と出力は、Google のモデルの改善に使われることがある。個人情報や機密の文書は送らない (有料枠に切り替えると使われなくなる)。
- 最近の Gemini は答える前に考える (思考) ため、思考のトークンも消費する。単純な処理なら `thinking_config=types.ThinkingConfig(thinking_budget=0)` で思考を止めると速くなる (止められないモデルもある)。
- `response.text` が `None` になることがある (安全フィルタで止められた、出力の上限に達した、など)。`response.candidates[0].finish_reason` で理由を確かめる。